In [ ]:
import math

import torch
from beartype import beartype
from jaxtyping import Float
from torch import Tensor, nn

In [ ]:
# Model params
num_heads = 1
d_model = 16
d_v = d_model // num_heads
d_k = d_v

assert d_model == d_v * num_heads

# Data params
batch = 1
length = 3

In [ ]:
X = torch.rand((batch, length, d_model), dtype=torch.float32)
print(f"X: {X.shape}")

Q = torch.rand((batch, length, d_k), dtype=torch.float32)
print(f"Q: {Q.shape}")

K = torch.rand((batch, length, d_k), dtype=torch.float32)
print(f"K: {K.shape}")

V = torch.rand((batch, length, d_v), dtype=torch.float32)
print(f"V: {V.shape}")

In [ ]:
@beartype
def scaled_dot_product_attn(
    Q: Float[Tensor, "b l d_k"],
    K: Float[Tensor, "b l d_k"],
    V: Float[Tensor, "b l d_v"],
) -> Float[Tensor, "b l d_v"]:
    # NOTE: This is only 1 head
    _batch, _length, d_k = K.shape

    factor = 1 / math.sqrt(d_k)

    K_t = torch.transpose(K, 1, 2)  # (b, d_k, l)
    scaled_scores = torch.bmm(Q, K_t) * factor  # (b, l, l)
    attn_weights = torch.softmax(scaled_scores, dim=2)  # (b, l, l)

    return torch.bmm(attn_weights, V)


@beartype
class AttentionHead(nn.Module):
    def __init__(self, d_model: int, d_k: int, d_v: int, qkv_bias: bool) -> None:
        super().__init__()
        self.W_q = nn.Linear(d_model, d_k, bias=qkv_bias)
        self.W_k = nn.Linear(d_model, d_k, bias=qkv_bias)
        self.W_v = nn.Linear(d_model, d_v, bias=qkv_bias)

    def forward(self, X: Float[Tensor, "b l d_model"]) -> Float[Tensor, "b l d_v"]:

        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        return scaled_dot_product_attn(Q, K, V)

In [ ]:
model = AttentionHead(d_model, d_k, d_v, qkv_bias=False)

out = model(X)
print(out.shape)

In [ ]:
scaled_dot_product_attn(Q, K, V)